# 🏛️ RAG Chatbot Hukum Indonesia
### Skilled Tier: Parent-Child Retriever + Semantic Router + Metadata UI
**Model:** Ganggas/legal-chatbot-llama3-indo-3b-final | **GPU:** T4 Colab

In [1]:
# ============================================================
# CELL 0 — Install Dependencies
# ============================================================
!pip install -q langchain==0.3.25
!pip install -q langchain-community==0.3.24
!pip install -q langchain-classic
!pip install -q "langchain-core>=0.3.59,<1.0.0"
!pip install -q langchain-text-splitters==0.3.8
!pip install -q langchain-huggingface==0.1.2
!pip install -q faiss-cpu==1.9.0
!pip install -q pypdf pymupdf
!pip install -q rank-bm25==0.2.2
!pip install -q sentence-transformers==3.4.1
!pip install -q transformers==4.46.3 accelerate bitsandbytes
!pip install -q gradio==5.30.0
!pip install -q unsloth
print("✅ Install selesai! Restart runtime dulu ya.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.25 requires langchain-core<1.0.0,>=0.3.58, but you have langchain-core 1.4.8 which is incompatible.
langchain 0.3.25 requires langchain-text-splitters<1.0.0,>=0.3.8, but you have langchain-text-splitters 1.1.2 which is incompatible.
langchain-huggingface 0.1.2 requires langchain-core<0.4.0,>=0.3.15, but you have langchain-core 1.4.8 which is incompatible.
langchain-community 0.3.24 requires langchain-core<1.0.0,>=0.3.59, but you have langchain-core 1.4.8 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.25 requires langchain-text-splitters<1.0.0,>=0.3.8, but you have langchain-text-splitters 1.1.2 which is incompatible.
langchain-text-splitt

In [2]:
# ============================================================
# CELL 1 — Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR  = "/content/drive/MyDrive/chatbot-rag-hukum/data_hukum"
FAISS_DIR = "/content/drive/MyDrive/chatbot-rag-hukum/faiss_db_skilled"

import os
print("PDF tersedia:")
for f in os.listdir(DATA_DIR):
    print(f"  - {f}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PDF tersedia:
  - PP Nomor 5 Tahun 2021.pdf
  - PP Nomor 35 Tahun 2021.pdf
  - PP Nomor 51 Tahun 2023.pdf
  - UU Nomor 6 Tahun 2023.pdf


In [3]:
# ============================================================
# CELL 2 — Import & Setup
# ============================================================
import os
import torch
import warnings
warnings.filterwarnings('ignore')
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_core.stores import InMemoryByteStore
from langchain_core.documents import Document
from langchain_classic.retrievers import ParentDocumentRetriever, EnsembleRetriever
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain.llms.base import LLM
from typing import Optional, List, Any
from unsloth.chat_templates import get_chat_template
import gradio as gr

try:
    hf_token = userdata.get('HF_TOKEN')
except:
    hf_token = os.getenv("HF_TOKEN")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"💻 Device: {device.upper()}")
print(f"🔑 HF_TOKEN: {hf_token is not None}")
if device == "cuda":
    print(f"🎮 GPU: {torch.cuda.get_device_name(0)}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
💻 Device: CUDA
🔑 HF_TOKEN: True
🎮 GPU: Tesla T4


In [4]:
# ============================================================
# CELL 3 — Load PDF + Metadata Enrichment
# ============================================================
PDF_FILES = [
    os.path.join(DATA_DIR, f)
    for f in os.listdir(DATA_DIR)
    if f.endswith('.pdf')
]

# Metadata per dokumen
DOC_META = {
    "PP Nomor 5 Tahun 2021.pdf":  {"peraturan": "PP No.5/2021",  "tahun": 2021, "topik": "Perizinan Berusaha"},
    "PP Nomor 35 Tahun 2021.pdf": {"peraturan": "PP No.35/2021", "tahun": 2021, "topik": "Ketenagakerjaan PKWT"},
    "PP Nomor 51 Tahun 2023.pdf": {"peraturan": "PP No.51/2023", "tahun": 2023, "topik": "Pengupahan"},
    "UU Nomor 6 Tahun 2023.pdf":  {"peraturan": "UU No.6/2023",  "tahun": 2023, "topik": "Cipta Kerja"},
}

all_documents = []
for path in PDF_FILES:
    loader = PyPDFLoader(path)
    docs = loader.load()
    fname = os.path.basename(path)
    extra = DOC_META.get(fname, {})
    for doc in docs:
        doc.metadata.update(extra)
        doc.metadata['source_file'] = fname
    all_documents.extend(docs)
    print(f"  {fname}: {len(docs)} halaman")

print(f"\n✅ Total: {len(all_documents)} halaman dimuat")

  PP Nomor 5 Tahun 2021.pdf: 739 halaman
  PP Nomor 35 Tahun 2021.pdf: 56 halaman
  PP Nomor 51 Tahun 2023.pdf: 27 halaman
  UU Nomor 6 Tahun 2023.pdf: 1127 halaman

✅ Total: 1949 halaman dimuat


In [5]:
# ============================================================
# CELL 4 — Parent-Child Retriever + BM25 Hybrid
# ============================================================

# Parent: chunk besar untuk konteks LLM
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=150,
)

# Child: chunk kecil untuk pencarian vektor (lebih presisi)
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
)

print("🔄 Memuat embedding model (BAAI/bge-m3)...")
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)
print("✅ Embedding model siap")

# Build FAISS dari child chunks
print("🧠 Membangun FAISS dari child chunks...")
child_chunks = child_splitter.split_documents(all_documents)
vectorstore = FAISS.from_documents(child_chunks, embedding_model)
print(f"✅ FAISS siap — {len(child_chunks)} child chunks")

# InMemoryByteStore untuk simpan parent chunks
docstore = InMemoryByteStore()

# Parent-Child Retriever
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=docstore,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
    search_type="similarity",
    search_kwargs={"k": 10}
)
parent_retriever.add_documents(all_documents)
print("✅ ParentDocumentRetriever siap")

# BM25 Retriever (keyword-based) dari parent chunks
parent_docs_for_bm25 = parent_splitter.split_documents(all_documents)
bm25_retriever = BM25Retriever.from_documents(parent_docs_for_bm25)
bm25_retriever.k = 10
print("✅ BM25 Retriever siap")

# Hybrid: BM25 (keyword) 40% + ParentDoc (semantic) 60%
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, parent_retriever],
    weights=[0.4, 0.6]
)
print("✅ Hybrid Retriever siap (BM25=40% + ParentDoc=60%)")

🔄 Memuat embedding model (BAAI/bge-m3)...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ Embedding model siap
🧠 Membangun FAISS dari child chunks...
✅ FAISS siap — 6543 child chunks
✅ ParentDocumentRetriever siap
✅ BM25 Retriever siap
✅ Hybrid Retriever siap (BM25=40% + ParentDoc=60%)


In [6]:
# ============================================================
# CELL 5 — Load LLM (LLaMA-3.2 3B Fine-tuned)
# ============================================================
model_id = "Ganggas/legal-chatbot-llama3-indo-3b-final"

tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token, use_fast=True)
tokenizer = get_chat_template(tokenizer, chat_template="llama-3")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("⏳ Loading model (4-bit)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
) if device == "cuda" else None

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
    token=hf_token
)
# Bersihkan generation config bawaan model
model.generation_config.temperature = None
model.generation_config.top_p       = None
model.generation_config.do_sample   = False

print("✅ Model siap!")
if device == "cuda":
    print(f"📊 VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

⏳ Loading model (4-bit)...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

✅ Model siap!
📊 VRAM: 4.52 GB


In [7]:
# ============================================================
# CELL 6 — Custom LLM Wrapper (Fix Chat Template)
# ============================================================
class LlamaRAGLLM(LLM):
    model: Any
    tokenizer: Any
    max_new_tokens: int = 400

    @property
    def _llm_type(self) -> str:
        return "llama_rag"

    def _call(self, prompt: str, stop: Optional[List[str]] = None, **kwargs) -> str:
        messages = [
            {"role": "system", "content": (
                "Kamu adalah asisten hukum Indonesia yang akurat dan terpercaya. "
                "Jawab HANYA berdasarkan konteks hukum yang diberikan. "
                "Jika informasi tidak ada di konteks, jawab: "
                "'Informasi tidak ditemukan dalam dokumen yang tersedia.' "
                "Jangan mengarang. Jawab dalam Bahasa Indonesia yang formal dan jelas."
            )},
            {"role": "user", "content": prompt}
        ]

        input_text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = self.tokenizer(input_text, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                do_sample=False,
                temperature=None,
                top_p=None,
                repetition_penalty=1.2,
                pad_token_id=self.tokenizer.eos_token_id
            )

        new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

llm = LlamaRAGLLM(model=model, tokenizer=tokenizer)
print("✅ Custom LLM wrapper siap!")

✅ Custom LLM wrapper siap!


In [8]:
# CELL 7 — Semantic Router (Keyword-based, lebih reliable)
KATA_KUNCI_HUKUM = [
    "hukum", "sanksi", "pasal", "undang", "peraturan", "perizinan",
    "berusaha", "usaha", "pekerja", "upah", "lembur", "pkwt", "kontrak",
    "kerja", "pesangon", "phk", "ketenagakerjaan", "pengupahan", "denda",
    "pidana", "administratif", "izin", "cipta", "pp", "uu", "ketentuan",
    "kewajiban", "hak", "perjanjian", "alih daya", "outsourcing"
]

def semantic_router(question: str) -> str:
    q_lower = question.lower()
    for kata in KATA_KUNCI_HUKUM:
        if kata in q_lower:
            return "hukum"
    return "umum"

# Test
print(f"'sanksi perizinan' → {semantic_router('sanksi perizinan berusaha')}")
print(f"'batas lembur' → {semantic_router('berapa batas lembur pekerja')}")
print(f"'resep masak' → {semantic_router('cara membuat nasi goreng')}")

'sanksi perizinan' → hukum
'batas lembur' → hukum
'resep masak' → umum


In [9]:
# ============================================================
# CELL 8 — RAG Pipeline Lengkap
# ============================================================
RAG_PROMPT_TEMPLATE = """Berikut adalah kutipan dari dokumen hukum resmi:

===KONTEKS===
{context}
===AKHIR KONTEKS===

Contoh format jawaban yang benar:
"Berdasarkan Pasal 12 ayat (2), sanksi administratif berupa peringatan tertulis, denda, hingga pencabutan izin usaha, sesuai tingkat pelanggaran yang dilakukan."

Sekarang, berdasarkan HANYA teks di KONTEKS di atas, jawab pertanyaan ini:
{question}

ATURAN KETAT:
- WAJIB kutip nomor pasal/ayat spesifik yang ada di konteks
- Jika informasi SEBAGIAN ada, jawab dengan info yang tersedia, JANGAN bilang "tidak ada informasi" lalu kontradiksi diri sendiri dengan menyebut angka
- Jika informasi BENAR-BENAR tidak ada sama sekali di konteks, tulis HANYA: "Tidak ditemukan dalam dokumen" — TANPA menambahkan keterangan lain setelahnya
- DILARANG menambahkan informasi dari luar konteks
- Jawab dalam Bahasa Indonesia, ringkas, konsisten, dan tidak kontradiktif

Jawaban berdasarkan dokumen:"""

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def format_sources(docs):
    seen = set()
    sources = []
    for doc in docs:
        peraturan = doc.metadata.get('peraturan', '?')
        page      = doc.metadata.get('page', '?')
        topik     = doc.metadata.get('topik', '')
        key = f"{peraturan}_p{page}"
        if key not in seen:
            seen.add(key)
            sources.append(f"📄 {peraturan} hal.{page} — {topik}")
    return "\n".join(sources)

def answer_legal_question(question: str):
    retrieved_docs = hybrid_retriever.invoke(question)[:5]   # balik ke 5, bukan 3

    if not retrieved_docs:
        return "Dokumen tidak ditemukan.", ""

    context_text = format_docs(retrieved_docs)
    sources_text = format_sources(retrieved_docs)

    # JANGAN truncate ke 800 char — itu motong pasal penting.
    # Llama 3.2 punya context window besar, gak perlu dipotong segini agresif.
    # Kalau mau ada limit, naikkan jadi ~3000-4000 karakter aja.
    if len(context_text) > 5000:
        context_text = context_text[:5000] + "..."

    prompt = RAG_PROMPT_TEMPLATE.format(context=context_text, question=question)
    answer = llm._call(prompt)
    return answer, sources_text

# Test pipeline
print("Test RAG Pipeline:")
ans, src = answer_legal_question("Apa saja sanksi administratif pelanggaran perizinan berusaha?")
print(f"Jawaban: {ans[:200]}")
print(f"Sumber:\n{src}")

Test RAG Pipeline:


Both `max_new_tokens` (=400) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Jawaban: Menurut Dokumen Konteks yang disediakan, sanksi administratif untuk pelanggaran perzinaan berusaha termasuk:

1. Penyitaan uang
2. Pembatasan aktivitas bisnis
3. Pembatalan izin
4. Pembatasan operasi

Sumber:
📄 PP No.5/2021 hal.225 — Perizinan Berusaha
📄 PP No.5/2021 hal.270 — Perizinan Berusaha
📄 PP No.5/2021 hal.207 — Perizinan Berusaha
📄 PP No.5/2021 hal.324 — Perizinan Berusaha
📄 UU No.6/2023 hal.236 — Cipta Kerja


In [10]:
# ============================================================
# CELL 9 — Full Chatbot Function (Router + RAG)
# ============================================================
def chatbot(question: str, history: list):
    if not question.strip():
        return "", history

    # Step 1: Semantic Router
    route = semantic_router(question)

    if route == "umum":
        answer = (
            "⚠️ Pertanyaan Anda tidak berkaitan dengan hukum atau peraturan "
            "dalam dokumen yang tersedia. Silakan ajukan pertanyaan seputar "
            "hukum ketenagakerjaan, perizinan berusaha, pengupahan, atau Cipta Kerja."
        )
        sources = ""
    else:
        # Step 2: RAG Pipeline
        answer, sources = answer_legal_question(question)

    # Format output dengan metadata
    full_response = answer
    if sources:
        full_response += f"\n\n---\n**📚 Referensi Dokumen:**\n{sources}"

    history.append((question, full_response))
    return "", history

print("✅ Chatbot function siap!")

✅ Chatbot function siap!


In [ ]:
# ============================================================
# CELL 10 — Gradio UI
# ============================================================

EXAMPLE_QUESTIONS = [
    "Apa saja sanksi administratif bagi pelaku usaha yang melanggar perizinan berusaha?",
    "Berapa batas maksimal waktu lembur pekerja per hari?",
    "Apa yang dimaksud dengan upah minimum dan bagaimana perhitungannya?",
    "Apa syarat perjanjian kerja waktu tertentu (PKWT)?",
    "Bagaimana ketentuan pesangon dalam UU Cipta Kerja?",
]

with gr.Blocks(title="Asisten Hukum Indonesia", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🏛️ Asisten Hukum Indonesia
    Chatbot RAG berbasis dokumen hukum: **PP No.5/2021**, **PP No.35/2021**, **PP No.51/2023**, **UU No.6/2023**
    """)

    chatbot_ui = gr.Chatbot(
        label="Percakapan",
        height=450,
        bubble_full_width=False
    )

    with gr.Row():
        msg_input = gr.Textbox(
            placeholder="Tanyakan seputar hukum ketenagakerjaan, perizinan, atau pengupahan...",
            label="Pertanyaan Anda",
            scale=4
        )
        send_btn = gr.Button("Kirim 🚀", variant="primary", scale=1)

    with gr.Row():
        clear_btn = gr.Button("🗑️ Hapus Chat", scale=1)

    gr.Markdown("### 💡 Contoh Pertanyaan")
    gr.Examples(
        examples=EXAMPLE_QUESTIONS,
        inputs=msg_input,
        label="Klik untuk coba:"
    )

    gr.Markdown("""
    ---
    > ⚠️ **Catatan:** Jawaban dihasilkan berdasarkan dokumen hukum yang tersedia.
    > Untuk keperluan hukum resmi, konsultasikan dengan ahli hukum.
    """)

    # Event handlers
    send_btn.click(
        fn=chatbot,
        inputs=[msg_input, chatbot_ui],
        outputs=[msg_input, chatbot_ui]
    )
    msg_input.submit(
        fn=chatbot,
        inputs=[msg_input, chatbot_ui],
        outputs=[msg_input, chatbot_ui]
    )
    clear_btn.click(
        fn=lambda: ([], ""),
        outputs=[chatbot_ui, msg_input]
    )

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://68d4f0275867d26ee6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Both `max_new_tokens` (=400) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=400) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_